# ARC-v0.22 — H3 Robustness Audit

**Purpose.** Stress-test the paper's primary utility-gap dynamics against alternative short-horizon summaries **without rerunning retrieval** and without changing any previously reported claim gate.

This is a post-confirmatory robustness audit. It is explicitly not used to retune the original experiments.

## Frozen contrasts

1. FEVER-BGE representation: IVF-PQ32 → IVF-SQ8 (ARC-v0.13)
2. FEVER-E5 representation: IVF-PQ32 → IVF-SQ8 (ARC-v0.18)
3. FEVER-E5 IVF search effort: nprobe 8 → 64 (ARC-v0.19)
4. FEVER-E5 HNSW search effort: efSearch 8 → 256 (ARC-v0.20c)

For each query-policy trajectory, let

\[
a_t = |u_H(t)-u_L(t)|, \quad t=0,\ldots,4.
\]

Existing endpoint:

\[
H3_{abs}=\operatorname{OLS}_t(a_t).
\]

Alternative summaries frozen here:

\[
R_1=a_4-a_0,
\]

\[
R_2=\frac{a_3+a_4}{2}-\frac{a_0+a_1}{2},
\]

\[
R_3=a_4-\min_t a_t.
\]

### Primary robustness criterion
The **sign of the query-averaged mean** of `R1` must agree with the sign of the query-averaged mean `H3abs` in all four frozen contrasts. `R2` is secondary corroboration; `R3` is descriptive only. Query-cluster bootstrap intervals are reported but do not redefine the gate after outcomes are observed.


In [ ]:
%pip install -q pyarrow pandas numpy

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, warnings
import numpy as np
import pandas as pd
from google.colab import drive

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 20260822
BOOTSTRAP_REPS = 10_000
EXPECTED_CONFIGS = 44
MAX_ROUNDS = 4

DRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_ROOT.is_dir():
    drive.mount("/content/drive")
assert DRIVE_ROOT.is_dir()
ARC_ROOT = DRIVE_ROOT / "rag-pq-checkpoints" / "arc-v0"
print("ARC root:", ARC_ROOT)


In [ ]:
# Resolve source runs by structural completeness only; do not inspect outcome values here.

def complete_v013(p):
    return p.is_dir() and len(list(p.glob("validation-*.parquet"))) == EXPECTED_CONFIGS

def complete_v018(p):
    return p.is_dir() and len(list((p / "validation").glob("validation-*.parquet"))) == EXPECTED_CONFIGS

def complete_v019(p):
    return p.is_dir() and len(list((p / "validation").glob("validation-*.parquet"))) == EXPECTED_CONFIGS

V013_ROOT = ARC_ROOT / "fever-boundary-external-replication-v013"
V018_ROOT = ARC_ROOT / "cross-encoder-fever-replication-v018"
V019_ROOT = ARC_ROOT / "cross-approximation-nprobe-replication-v019"
V020C_ROOT = ARC_ROOT / "hnsw-mechanism-replication-v020c"

pref13 = V013_ROOT / "20260817-140640"
V013_RUN = pref13 if complete_v013(pref13) else sorted([p for p in V013_ROOT.iterdir() if complete_v013(p)], reverse=True)[0]

pref18 = V018_ROOT / "20260819-015645"
V018_RUN = pref18 if complete_v018(pref18) else sorted([p for p in V018_ROOT.iterdir() if complete_v018(p)], reverse=True)[0]

v19_candidates = sorted([p for p in V019_ROOT.iterdir() if complete_v019(p)], reverse=True)
assert v19_candidates, "No complete v0.19 run found"
V019_RUN = v19_candidates[0]

HNSW_VAL = V020C_ROOT / "v020c_validation_endpoints.parquet"
assert HNSW_VAL.is_file(), HNSW_VAL

print("v0.13:", V013_RUN)
print("v0.18:", V018_RUN)
print("v0.19:", V019_RUN)
print("v0.20c:", HNSW_VAL)


In [ ]:
# Freeze this audit protocol BEFORE loading outcome values.
OUT_ROOT = ARC_ROOT / "h3-robustness-v022"
OUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
OUT = OUT_ROOT / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)

PROTOCOL = {
    "status": "ARC_V022_H3_ROBUSTNESS_PROTOCOL_FROZEN_BEFORE_OUTCOME_LOADING",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "bootstrap_reps": BOOTSTRAP_REPS,
    "sampling_unit": "query; all policy realizations remain clustered within query",
    "contrasts": {
        "BGE_rep": str(V013_RUN),
        "E5_rep": str(V018_RUN),
        "E5_nprobe": str(V019_RUN),
        "E5_HNSW": str(HNSW_VAL),
    },
    "metrics": {
        "H3abs": "OLS slope over a_t=abs(u_H(t)-u_L(t)), t=0..4",
        "R1": "a4-a0",
        "R2": "mean(a3,a4)-mean(a0,a1)",
        "R3": "a4-min(a0..a4)",
    },
    "primary_gate": "sign(query-averaged mean R1) == sign(query-averaged mean H3abs) for all 4 contrasts",
    "secondary": "R2 sign and 95% query-cluster bootstrap CIs",
    "descriptive": "R3",
    "no_retrieval_rerun": True,
    "no_retuning": True,
}
PROTOCOL_PATH = OUT / "v022_h3_robustness_protocol.json"
PROTOCOL_PATH.write_text(json.dumps(PROTOCOL, indent=2, sort_keys=True), encoding="utf-8")
print("FROZEN:", PROTOCOL_PATH)


In [ ]:
GROUP_COLS = ["query_id", "config_key"]

def ols_slope(y):
    y = np.asarray(y, dtype=np.float64)
    x = np.arange(len(y), dtype=np.float64)
    return float(np.polyfit(x, y, 1)[0])

def trajectory_to_metrics(df):
    required = {"query_id", "iteration", "abs_utility_gap"}
    missing = required - set(df.columns)
    assert not missing, f"Missing columns: {missing}"
    if "config_key" not in df.columns:
        # v0.13 files carry config_key; this fallback preserves file-level identity if needed.
        df = df.copy()
        df["config_key"] = "unknown"
    rows=[]
    for (qid, ck), g in df.groupby(GROUP_COLS, dropna=False, sort=False):
        g = g.sort_values("iteration")
        a = g["abs_utility_gap"].to_numpy(np.float64)
        assert len(a) == MAX_ROUNDS + 1, (qid, ck, len(a))
        rows.append({
            "query_id": str(qid), "config_key": str(ck),
            "H3abs": ols_slope(a),
            "R1": float(a[4]-a[0]),
            "R2": float((a[3]+a[4])/2.0 - (a[0]+a[1])/2.0),
            "R3": float(a[4]-a.min()),
            "a0": float(a[0]), "a1": float(a[1]), "a2": float(a[2]), "a3": float(a[3]), "a4": float(a[4]),
        })
    return pd.DataFrame(rows)

def load_many(paths):
    frames=[]
    for p in sorted(paths):
        d = pd.read_parquet(p)
        if "config_key" not in d.columns:
            d = d.copy(); d["config_key"] = p.stem
        frames.append(d)
    assert frames
    return pd.concat(frames, ignore_index=True)

def hnsw_endpoint_to_metrics(df):
    need = {"query_id", "H3abs", "a0", "a1", "a2", "a3", "a4"}
    missing = need - set(df.columns)
    assert not missing, f"HNSW endpoint missing columns: {missing}"
    out = df.copy()
    if "config_key" not in out.columns:
        fam = out.get("family", pd.Series(["x"]*len(out))).astype(str)
        alpha = out.get("alpha", pd.Series([np.nan]*len(out))).astype(str)
        k = out.get("k", pd.Series([np.nan]*len(out))).astype(str)
        temp = out.get("temperature", pd.Series([np.nan]*len(out))).astype(str)
        out["config_key"] = fam + "|a=" + alpha + "|k=" + k + "|t=" + temp
    out["R1"] = out["a4"] - out["a0"]
    out["R2"] = (out["a3"] + out["a4"])/2.0 - (out["a0"] + out["a1"])/2.0
    out["R3"] = out["a4"] - out[["a0","a1","a2","a3","a4"]].min(axis=1)
    return out[["query_id","config_key","H3abs","R1","R2","R3","a0","a1","a2","a3","a4"]].copy()


In [ ]:
# Load outcomes only after protocol freeze.
raw_bge = load_many(V013_RUN.glob("validation-*.parquet"))
raw_e5r = load_many((V018_RUN / "validation").glob("validation-*.parquet"))
raw_np  = load_many((V019_RUN / "validation").glob("validation-*.parquet"))
raw_hn  = pd.read_parquet(HNSW_VAL)

metrics = {
    "BGE_rep": trajectory_to_metrics(raw_bge),
    "E5_rep": trajectory_to_metrics(raw_e5r),
    "E5_nprobe": trajectory_to_metrics(raw_np),
    "E5_HNSW": hnsw_endpoint_to_metrics(raw_hn),
}

for name, d in metrics.items():
    print(name, d.shape, "queries", d.query_id.nunique(), "configs", d.config_key.nunique())
    assert d.query_id.nunique() > 3000
    assert d.config_key.nunique() == EXPECTED_CONFIGS


In [ ]:
def query_average(d):
    return d.groupby("query_id", as_index=False)[["H3abs","R1","R2","R3"]].mean()

def bootstrap_mean_ci(x, reps=BOOTSTRAP_REPS, seed=SEED):
    x = np.asarray(x, dtype=np.float64)
    rng = np.random.default_rng(seed)
    n = len(x)
    vals = np.empty(reps, dtype=np.float64)
    # Chunk bootstrap indices to keep memory bounded.
    chunk = 250
    done=0
    while done < reps:
        m=min(chunk, reps-done)
        idx=rng.integers(0,n,size=(m,n))
        vals[done:done+m]=x[idx].mean(axis=1)
        done += m
    return float(x.mean()), float(np.quantile(vals,0.025)), float(np.quantile(vals,0.975))

def sgn(x, tol=0.0):
    return 1 if x>tol else (-1 if x< -tol else 0)

rows=[]
for i,(name,d) in enumerate(metrics.items()):
    q=query_average(d)
    stats={}
    for j,m in enumerate(["H3abs","R1","R2","R3"]):
        stats[m]=bootstrap_mean_ci(q[m].to_numpy(), seed=SEED+100*i+j)
    hmean=stats["H3abs"][0]; r1mean=stats["R1"][0]
    rows.append({
        "setting": name,
        "n_queries": len(q),
        "H3abs_mean": hmean, "H3abs_ci_lo": stats["H3abs"][1], "H3abs_ci_hi": stats["H3abs"][2],
        "R1_mean": r1mean, "R1_ci_lo": stats["R1"][1], "R1_ci_hi": stats["R1"][2],
        "R2_mean": stats["R2"][0], "R2_ci_lo": stats["R2"][1], "R2_ci_hi": stats["R2"][2],
        "R3_mean": stats["R3"][0], "R3_ci_lo": stats["R3"][1], "R3_ci_hi": stats["R3"][2],
        "H3_sign": sgn(hmean), "R1_sign": sgn(r1mean),
        "primary_sign_preserved": bool(sgn(hmean)==sgn(r1mean)),
        "R1_ci_excludes_zero": bool(stats["R1"][1]>0 or stats["R1"][2]<0),
        "R2_sign_matches_H3": bool(sgn(stats["R2"][0])==sgn(hmean)),
    })

summary=pd.DataFrame(rows)
display(summary)


In [ ]:
PRIMARY_PASS = bool(summary["primary_sign_preserved"].all())
SECONDARY_R2_ALL = bool(summary["R2_sign_matches_H3"].all())
R1_CI_ALL = bool(summary["R1_ci_excludes_zero"].all())

verdict = "H3_ROBUSTNESS_PRIMARY_PASS" if PRIMARY_PASS else "H3_ROBUSTNESS_PRIMARY_FAIL"
print("Primary:", verdict)
print("R2 sign corroboration all four:", SECONDARY_R2_ALL)
print("R1 95% CI excludes zero all four:", R1_CI_ALL)

summary.to_csv(OUT / "v022_h3_robustness_summary.csv", index=False)
for name,d in metrics.items():
    d.to_parquet(OUT / f"v022_{name}_query_policy_metrics.parquet", index=False)

REPORT = {
    "status": "ARC_V022_H3_ROBUSTNESS_COMPLETE",
    "primary_pass": PRIMARY_PASS,
    "secondary_R2_sign_all": SECONDARY_R2_ALL,
    "R1_ci_excludes_zero_all": R1_CI_ALL,
    "verdict": verdict,
    "summary": summary.to_dict(orient="records"),
    "interpretation_rule": {
        "if_pass": "Core representation-expansion/search-effort-contraction sign map is not an artifact of OLS H3abs under the frozen R1 endpoint.",
        "if_fail": "Do not claim OLS-independent direction; report which contrast changes sign and scope the paper accordingly.",
    },
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}
(OUT / "v022_h3_robustness_report.json").write_text(json.dumps(REPORT, indent=2), encoding="utf-8")
print("Saved:", OUT)


## Paper-facing interpretation

Do **not** paste a stronger claim merely because the audit passes. The safe paper-facing addition is one sentence such as:

> The representation-expansion/search-effort-contraction sign pattern is preserved when utility-gap dynamics are summarized by final-minus-initial separation rather than OLS slope (post-confirmatory robustness audit).

If any contrast changes sign under `R1`, retain that result and weaken the H3 operational claim instead of tuning the alternative endpoint.
